# SWMAL Exercise

## Pipelines

In [1]:
%matplotlib inline

import sys
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def LoadDataFromL01():
    filename = os.path.join("Data", "itmal_l01_data.pkl")
    with open(f"{filename}", "rb") as f:
        (X, y) = pickle.load(f)
        return X, y

X, y = LoadDataFromL01()

print(f"X.shape={X.shape},  y.shape={y.shape}")

assert X.shape[0] == y.shape[0]
assert X.ndim == 2
assert y.ndim == 1  # did a y.ravel() before saving to picke file
assert X.shape[0] == 29

# re-create plot data (not stored in the Pickel file)
m = np.linspace(0, 60000, 1000)
M = np.empty([m.shape[0], 1])
M[:, 0] = m

print("OK")

X.shape=(29, 1),  y.shape=(29,)
OK


In [2]:
# Setup the MLP and lin. regression again..

def isNumpyData(t: np.ndarray, expected_ndim: int):
    assert isinstance(expected_ndim, int), f"input parameter 'expected_ndim' is not an integer but a '{type(expected_ndim)}'"
    assert expected_ndim>=0, f"expected input parameter 'expected_ndim' to be >=0, got {expected_ndim}"
    if t is None:
        print("input parameter 't' is None", file=sys.stderr)
        return False
    if not isinstance(t, np.ndarray):
        print("excepted numpy.ndarray got type '{type(t)}'", file=sys.stderr)
        return False
    if not t.ndim==expected_ndim:
        print("expected ndim={expected_ndim} but found {t.ndim}", file=sys.stderr)
        return False
    return True

def PlotModels(model1, model2, X: np.ndarray, y: np.ndarray, name_model1: str, name_model2: str):
    
    # NOTE: local function is such a nifty feature of Python!
    def CalcPredAndScore(model, X: np.ndarray, y: np.ndarray,):
        assert isNumpyData(X, 2) and isNumpyData(y, 1) and X.shape[0]==y.shape[0]
        y_pred_model = model.predict(X)
        score_model = r2_score(y, y_pred_model) # call r2
        return y_pred_model, score_model    

    assert isinstance(name_model1, str) and isinstance(name_model2, str)

    y_pred_model1, score_model1 = CalcPredAndScore(model1, X, y)
    y_pred_model2, score_model2 = CalcPredAndScore(model2, X, y)

    plt.plot(X, y_pred_model1, "r.-")
    plt.plot(X, y_pred_model2, "kx-")
    plt.scatter(X, y)
    plt.xlabel("GDP per capita")
    plt.ylabel("Life satisfaction")
    plt.legend([name_model1, name_model2, "X OECD data"])

    l = max(len(name_model1), len(name_model2))
    
    print(f"{(name_model1).rjust(l)}.score(X, y)={score_model1:0.2f}")
    print(f"{(name_model2).rjust(l)}.score(X, y)={score_model2:0.2f}")

# lets make a linear and MLP regressor and redo the plots
mlp = MLPRegressor(hidden_layer_sizes=(10, ),
                   solver='adam',
                   activation='relu',
                   tol=1E-5,
                   max_iter=100000,
                   verbose=False)
linreg = LinearRegression()

mlp.fit(X, y)
linreg.fit(X, y)

print("The MLP may mis-fit the data, seen in the, sometimes, bad R^2 score..\n")
PlotModels(linreg, mlp, X, y, "lin.reg", "MLP")
print("\nOK")

The MLP may mis-fit the data, seen in the, sometimes, bad R^2 score..

lin.reg.score(X, y)=0.73
    MLP.score(X, y)=-5.24

OK


### Qa) Create a Min/max scaler for the MLP

We scale **X** to **[0,1]** before training the MLP. This makes the features share a common range which stabilizes gradients and helps MLPs train. We fit the scaler on the train split, then reuse it for val/test. Sources: [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)  [Neural networks: supervised](https://scikit-learn.org/stable/modules/neural_networks_supervised.html).

In [3]:
def minmax_scaler(X):
    X_min = X.min()
    X_max = X.max()
    X_minmax_scaled = (X - X_min) / (X_max - X_min)
    
    return X_minmax_scaled

X_scaled_myfunc=minmax_scaler(X)

# Retrain
mlp.fit(X_scaled_myfunc, y)
linreg.fit(X_scaled_myfunc, y)

# Replot
print("After manually scaling the input data to [0;1]:\n")
PlotModels(linreg, mlp, X_scaled_myfunc, y, "lin.reg", "MLP")

After manually scaling the input data to [0;1]:

lin.reg.score(X, y)=0.73
    MLP.score(X, y)=0.72


Scaling X to [0,1] fixed the MLP’s misfit: it now matches linear regression (**R² ≈ 0.73**). So the issue was scale, not model capacity.

### Qb) Scikit-learn Pipelines

We wrap the scaler and the MLP in one `Pipeline`. That way `fit` runs the steps in order, and `predict` always applies the **same** scaling as training. It keeps the code tidy and helps avoid leakage bugs. Source: [Scikit-learn’s pipeline guide](https://scikit-learn.org/stable/modules/compose.html#pipeline).

In [4]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

# Plain MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

mlp_scaled = MLPRegressor(hidden_layer_sizes=(10,),
                          solver='adam', activation='relu',
                          tol=1e-5, max_iter=100000, verbose=False)
linreg_scaled = LinearRegression()

mlp_scaled.fit(X_scaled, y)
linreg_scaled.fit(X_scaled, y)

print("Scaled (manual via MinMaxScaler) results:")
PlotModels(linreg_scaled, mlp_scaled, X_scaled, y, "lin.reg (scaled)", "MLP (scaled)")

# Composite estimator
mlp_pipe = Pipeline([
    ("scale", MinMaxScaler()),
    ("mlp", MLPRegressor(hidden_layer_sizes=(10,),
                         solver='adam', activation='relu',
                         tol=1e-5, max_iter=100000, verbose=False))
])

mlp_pipe.fit(X, y)
y_pred_pipe = mlp_pipe.predict(X)

print("\nPipeline results (scale -> MLP):")
print(f"MLP Pipe R^2 = {r2_score(y, y_pred_pipe):0.2f}")


Scaled (manual via MinMaxScaler) results:
lin.reg (scaled).score(X, y)=0.73
    MLP (scaled).score(X, y)=0.72

Pipeline results (scale -> MLP):
MLP Pipe R^2 = 0.72


Using `MinMaxScaler` and wrapping it with the MLP in a `Pipeline` gives the around the performance (**R² ≈ 0.72–0.73**) and a cleaner, safer workflow. The small 0.01 dip is fine; the good thing is that it is more reproduceble and there is no scaling leaks.

### Qc) Outliers and the Min-max Scaler vs. the Standard Scaler

**The problem with Min–Max struggles and outliers.**  
Min–max maps data to [0,1] using the smallest and largest values. If one value is extreme, it becomes 1.0 and squeezes almost all other points into a tiny range. The model then learns poorly from the “squished” bulk.

**Is StandardScaler better?**  
Usually yes. Standardization (mean 0, std 1) which is less fragile. But it’s not perfect. Big outliers can still affect the mean/std.

**Sources:**  
- [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)  
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [Scaling and normalization](https://scikit-learn.org/stable/modules/preprocessing.html)


In [5]:
from sklearn.preprocessing import StandardScaler


def eval_pipe(name, scaler, X, y):
    mlp = Pipeline([
        ("scale", scaler),
        ("mlp", MLPRegressor(hidden_layer_sizes=(10,), solver="adam",
                             activation="relu", tol=1e-5, max_iter=100000, verbose=False))
    ])
    lin = Pipeline([("scale", scaler), ("lin", LinearRegression())])

    mlp.fit(X, y); lin.fit(X, y)
    r2_mlp = r2_score(y, mlp.predict(X))
    r2_lin = r2_score(y, lin.predict(X))
    print(f"{name:>12} | MLP R^2={r2_mlp:0.2f} | Lin R^2={r2_lin:0.2f}")

print("Scaler comparison on the same data (fit on X, for demo):")
eval_pipe("MinMax",   MinMaxScaler(),   X, y)
eval_pipe("Standard", StandardScaler(), X, y)


Scaler comparison on the same data (fit on X, for demo):
      MinMax | MLP R^2=0.73 | Lin R^2=0.73
    Standard | MLP R^2=0.78 | Lin R^2=0.73


Standardization clearly helped the MLP here (**R² 0.79** vs. **0.73** with MinMax), while linear regression stayed the same (**0.73**) because scaling doesn’t change its fit quality.

### Qd) Modify the MLP Hyperparameters

We run a small hyperparameter sweep for the MLP to see how few neurons we can use and still get a strong fit. The script standardizes `X`, then tries different widths (neurons), activations (`relu`, `tanh`, `logistic`), and solvers (`adam`, `lbfgs`, `sgd`). For each combo, we fit the model, compute R², list the top scores, and report the smallest network that reaches a target R². (Sources: [MLPRegressor docs](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html), [Neural networks: supervised](https://scikit-learn.org/stable/modules/neural_networks_supervised.html))

In [ ]:
import pandas as pd

def run_mlp_sweep(X, y, neurons_list=(1,2,3,5,8,10,15,20,30),
                  activations=("relu","tanh","logistic"),
                  solvers=("adam","lbfgs","sgd"),
                  target_r2=0.75,
                  max_iter=200000,
                  random_state=42):
    rows = []
    for act in activations:
        for sol in solvers:
            for n in neurons_list:
                kwargs = dict(hidden_layer_sizes=(n,),
                              activation=act,
                              solver=sol,
                              max_iter=max_iter,
                              tol=1e-6,
                              random_state=random_state)
                if sol == "sgd":
                    kwargs.update(dict(learning_rate_init=1e-2, momentum=0.9, nesterovs_momentum=True))
                model = Pipeline([
                    ("scale", StandardScaler()),
                    ("mlp", MLPRegressor(**kwargs))
                ])
                model.fit(X, y)
                y_pred = model.predict(X)
                r2 = r2_score(y, y_pred)
                rows.append(dict(neurons=n, activation=act, solver=sol, r2=r2))
    df = pd.DataFrame(rows).sort_values(["activation","solver","neurons"]).reset_index(drop=True)

    best_idx = df["r2"].idxmax()
    best = df.loc[best_idx].to_dict()

    meets = []
    for act in activations:
        for sol in solvers:
            sub = df[(df.activation==act) & (df.solver==sol)]
            sub_meet = sub[sub.r2 >= target_r2]
            if len(sub_meet):
                meets.append(sub_meet.sort_values("neurons").iloc[0])
    meets_df = pd.DataFrame(meets).sort_values(["r2","neurons"], ascending=[False, True])

    return df, best, meets_df

df_qd, best_qd, meets_qd = run_mlp_sweep(X, y)

print("Top 10 configs by R^2:")
display(df_qd.sort_values("r2", ascending=False).head(10))

print("\nBest overall config:")
print(best_qd)

print("\nSmallest networks that hit target R^2 (default 0.75), by (activation, solver):")
display(meets_qd if len(meets_qd) else "No configs reached the target")


Top 10 configs by R^2:


,neurons,activation,solver,r2
71,30,tanh,lbfgs,0.997070
69,15,tanh,lbfgs,0.988211
70,20,tanh,lbfgs,0.987565
16,20,logistic,lbfgs,0.984780
15,15,logistic,lbfgs,0.979554
17,30,logistic,lbfgs,0.978976
13,8,logistic,lbfgs,0.934920
14,10,logistic,lbfgs,0.904275
68,10,tanh,lbfgs,0.882577
12,5,logistic,lbfgs,0.873759



Best overall config:
{'neurons': 30, 'activation': 'tanh', 'solver': 'lbfgs', 'r2': 0.9970698678189469}

Smallest networks that hit target R^2 (default 0.75), by (activation, solver):


,neurons,activation,solver,r2
50,10,relu,sgd,0.815845
76,8,tanh,sgd,0.801711
63,1,tanh,lbfgs,0.798252
9,1,logistic,lbfgs,0.798252
56,3,tanh,adam,0.795598
1,2,logistic,adam,0.793564
37,2,relu,lbfgs,0.764624
29,3,relu,adam,0.760653


`tanh + lbfgs` with ~30 neurons topped the list (very high R²), and even tiny networks (1–3 neurons) can hit ~0.76–0.80 with the right settings. `lbfgs` often works well on small datasets, while `sgd` typically needs a bit more width. (Source: [MLPRegressor: activations/solvers](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html))

REVISIONS||
:-|:-|
2020-10-15| CEF, initial. 
2020-10-21| CEF, added Standard Scaler Q.
2020-11-17| CEF, removed orhpant text in Qa (moded to Qc).
2021-02-10| CEF, updated for ITMAL F21.
2021-11-08| CEF, updated print info.
2021-02-10| CEF, updated for SWMAL F22.
2023-02-19| CEF, updated for SWMAL F23, adjuste page numbers for 3rd.ed.
2023-02-21| CEF, added types, rewrote CalcPredAndScore and added isNumpyData.
2024-09-11| CEF, updated page refefences.